In [15]:
import os
import json
import pymupdf
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
from IPython.display import display, IFrame, Markdown, HTML
from IPython.display import Image as DisplayImage
from PIL import Image as PILImage, ImageDraw
from typing import List

In [16]:
from landingai_ade import LandingAIADE
from landingai_ade.types import ParseResponse, ExtractResponse

In [17]:
_ = load_dotenv(override=True)

In [18]:
client = LandingAIADE()
print("Authenticated client initialized")

Authenticated client initialized


In [19]:
from pydantic import BaseModel, Field
from typing import List, Literal


class DocType(BaseModel):
    type: Literal["timetable", "other"] = Field(
        description="The type of document being analyzed.",
        title="Document Type",
    )

class TimetableEntry(BaseModel):
    group_number: str = Field(description="The group number of the course.",
                               title="Group Number")
    course_name: str = Field(description="The name of the course.", 
                           title="Course Name")
    course_code: str = Field(description="The code of the course.", 
                                title="Course Code")
    course_type: str = Field(description="The type of the course.", 
                          title="Course Type")
    instructor_name: str = Field(description="The name of the instructor.", 
                                title="Instructor Name")
    class_number: str = Field(description="The number of the class.", 
                                title="Class Number")
    time: str = Field(description="The time of the class.", 
                                title="Time")
    day: str = Field(description="The day of the class.", 
                                title="Day")

class TimetableSchema(BaseModel):
    entries: List[TimetableEntry] = Field(
        description="List of all timetable entries/courses",
        title="Timetable Entries"
    )
    

In [ ]:
# Convert Pydantic models to JSON schema for Landing AI
import json

# Try to use Landing AI's utility function
try:
    from landingai_ade.utils import pydantic_to_json_schema
    timetable_json_schema = pydantic_to_json_schema(TimetableSchema)
except ImportError:
    # Fallback: use Pydantic's built-in method and ensure it's a dict
    def pydantic_to_json_schema(model):
        schema = model.model_json_schema()
        # Ensure it's a proper dict, not a string
        if isinstance(schema, str):
            schema = json.loads(schema)
        return schema
    
    timetable_json_schema = pydantic_to_json_schema(TimetableSchema)

# Ensure schema is a dict (not a string)
if isinstance(timetable_json_schema, str):
    timetable_json_schema = json.loads(timetable_json_schema)

# Debug: Verify schema format
print("✅ Schema prepared")
print(f"   Schema type: {type(timetable_json_schema)}")
print(f"   Schema keys: {list(timetable_json_schema.keys()) if isinstance(timetable_json_schema, dict) else 'Not a dict'}")

In [ ]:
input_folder = Path("time tables/")
output_folder = Path("output")
output_folder.mkdir(exist_ok=True)

# Process each document in the folder
for document in input_folder.iterdir():

    # 🔥 Skip directories so ADE doesn't try to parse them
    if document.is_dir():
        continue
        
    print(f"\n{'='*60}")
    print(f"Processing document: {document.name}")
    print(f"{'='*60}\n")

    # Step 1: Parse the document to extract layout and content
    print("Step 1: Parsing document with DPT-2 model...")
    parse_result: ParseResponse = client.parse(
        document=document,
        split="page",
        model="dpt-2-latest"
    )
    print("✅ Parsing completed.")
    print()
    
    # Combine markdown from all pages
    all_markdown = "\n\n".join([split.markdown for split in parse_result.splits])
    print(f"Extracted markdown length: {len(all_markdown)} characters")
    print()
    
    # Step 2: Extract timetable data directly
    print("Step 2: Extracting timetable data...")
    try:
        # Try multiple approaches to pass the schema
        # Approach 1: Try with the JSON schema dict
        schema_dict = timetable_json_schema
        if isinstance(schema_dict, str):
            schema_dict = json.loads(schema_dict)
        
        # Debug: Print schema structure
        print(f"   Schema type: {type(schema_dict)}")
        print(f"   Schema top-level keys: {list(schema_dict.keys())[:5]}")
        
        # Try extraction - the client may need the schema in a specific format
        # If this fails, it might be a client library issue
        extraction_result: ExtractResponse = client.extract(
            schema=schema_dict,
            markdown=all_markdown
        )
        
        # Get the extracted entries
        extraction = extraction_result.extraction if hasattr(extraction_result, 'extraction') else {}
        timetable_data = extraction.get("entries", [])
        
        if not timetable_data:
            print(f"⚠️  Warning: No timetable entries found in {document.name}")
            print(f"   Extraction keys: {list(extraction.keys())}")
            # Print the full extraction to debug
            if extraction:
                print(f"   Full extraction structure: {json.dumps(extraction, indent=2)[:500]}")
            continue
        
        print(f"✅ Found {len(timetable_data)} timetable entries")
        print()
        
        # Step 3: Convert to DataFrame and save as CSV
        print("Step 3: Converting to CSV...")
        df = pd.DataFrame(timetable_data)
        
        # Display the DataFrame
        print("\nExtracted Timetable Data:")
        display(df)
        
        # Save to CSV
        document_stem = Path(document).stem
        csv_filename = output_folder / f"{document_stem}_timetable.csv"
        df.to_csv(csv_filename, index=False, encoding='utf-8')
        print(f"\n✅ Timetable saved to: {csv_filename}")
        print(f"   Total entries: {len(df)}")
        print()
        
    except Exception as e:
        print(f"❌ Error extracting timetable data: {str(e)}")
        import traceback
        traceback.print_exc()
        print()


Processing document: pop.jpg

Step 1: Parsing document with DPT-2 model...
✅ Parsing completed.

Extracted markdown length: 2680 characters

Step 2: Extracting timetable data...
❌ Error extracting timetable data: Error code: 422 - {'message': '[{\'type\': \'missing\', \'loc\': (\'body\', \'schema\'), \'msg\': \'Field required\', \'input\': {\'markdown\': UploadFile(filename=\'upload\', size=2683, headers=Headers({\'content-disposition\': \'form-data; name="markdown"; filename="upload"\', \'content-type\': \'application/octet-stream\'})), \'schema[$defs][TimetableEntry][properties][group_number][description]\': \'The group number of the course.\', \'schema[$defs][TimetableEntry][properties][group_number][title]\': \'Group Number\', \'schema[$defs][TimetableEntry][properties][group_number][type]\': \'string\', \'schema[$defs][TimetableEntry][properties][course_name][description]\': \'The name of the course.\', \'schema[$defs][TimetableEntry][properties][course_name][title]\': \'Course N

Traceback (most recent call last):
  File "C:\Users\Mouhanned\AppData\Local\Temp\ipykernel_4080\3769219990.py", line 34, in <module>
    extraction_result: ExtractResponse = client.extract(
                                         ^^^^^^^^^^^^^^^
  File "c:\Users\Mouhanned\AppData\Local\Programs\Python\Python312\Lib\site-packages\landingai_ade\_client.py", line 362, in extract
    result = self.post(
             ^^^^^^^^^^
  File "c:\Users\Mouhanned\AppData\Local\Programs\Python\Python312\Lib\site-packages\landingai_ade\_base_client.py", line 1242, in post
    return cast(ResponseT, self.request(cast_to, opts, stream=stream, stream_cls=stream_cls))
                           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Mouhanned\AppData\Local\Programs\Python\Python312\Lib\site-packages\landingai_ade\_base_client.py", line 1044, in request
    raise self._make_status_error_from_response(err.response) from None
landingai_ade.UnprocessableEntityError

In [ ]:
# This cell is no longer needed - all processing is done in Cell 6
# The timetable extraction and CSV conversion now happens directly in Cell 6
print("✅ Timetable processing is now integrated into Cell 6.")
print("   Run Cell 6 to process all timetable images and generate CSV files.")